In [223]:
!pip install fastdtw scikit-learn imbalanced-learn pandas numpy scipy


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [224]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.combine import SMOTEENN # [ĐÃ SỬA]: Dùng SMOTEENN để làm sạch ranh giới
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import warnings

warnings.filterwarnings('ignore')

**1. PREPROCESSING**

In [225]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def preprocess_data(df):
    df = df.sort_values(by=['Joint_Name', 'Frame']).reset_index(drop=True)
    
    mask = df['Visibility'] < 0.5
    df.loc[mask, ['X', 'Y', 'Z']] = np.nan
    
    df[['X', 'Y', 'Z']] = df.groupby('Joint_Name')[['X', 'Y', 'Z']].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df.fillna(0, inplace=True) 
    
    df['X_norm'] = min_max_scale(df['X'])
    df['Y_norm'] = min_max_scale(df['Y'])
    df['Z_norm'] = min_max_scale(df['Z'])
    
    return df.sort_values(by=['Frame', 'Joint_Name']).reset_index(drop=True)

**2. TRÍCH XUẤT ĐẶC TRƯNG & CĂN CHỈNH THỜI GIAN (DTW)**

In [226]:
def align_and_extract_features(df_student, df_teacher):
    def get_frame_vectors(df):
        df_unique = df.drop_duplicates(subset=['Frame', 'Joint_Name'])
        wide = df_unique.pivot(index='Frame', columns='Joint_Name', values=['X_norm', 'Y_norm', 'Z_norm'])
        wide.fillna(0, inplace=True)
        return wide.values, wide.index.tolist()

    vec_s, frames_s = get_frame_vectors(df_student)
    vec_t, frames_t = get_frame_vectors(df_teacher)

    distance, path = fastdtw(vec_s, vec_t, dist=euclidean)

    aligned_pairs = [{'Frame_s': frames_s[s_idx], 'Frame_t': frames_t[t_idx]} for s_idx, t_idx in path]
    df_path = pd.DataFrame(aligned_pairs)

    df_s_aligned = pd.merge(df_path, df_student, left_on='Frame_s', right_on='Frame').drop(columns=['Frame'])
    df_t_aligned = pd.merge(df_path, df_teacher, left_on='Frame_t', right_on='Frame').drop(columns=['Frame'])
    
    df_merged = pd.merge(
        df_s_aligned, 
        df_t_aligned, 
        on=['Frame_s', 'Frame_t', 'Joint_Name'], 
        suffixes=('_s', '_t')
    )
    
    df_merged['Joint_Error'] = np.sqrt(
        (df_merged['X_norm_s'] - df_merged['X_norm_t'])**2 + 
        (df_merged['Y_norm_s'] - df_merged['Y_norm_t'])**2 + 
        (df_merged['Z_norm_s'] - df_merged['Z_norm_t'])**2
    )
    
    dot = (df_merged['X_norm_s']*df_merged['X_norm_t']) + \
          (df_merged['Y_norm_s']*df_merged['Y_norm_t']) + \
          (df_merged['Z_norm_s']*df_merged['Z_norm_t'])
    
    norm_s = np.sqrt(df_merged['X_norm_s']**2 + df_merged['Y_norm_s']**2 + df_merged['Z_norm_s']**2)
    norm_t = np.sqrt(df_merged['X_norm_t']**2 + df_merged['Y_norm_t']**2 + df_merged['Z_norm_t']**2)
    
    df_merged['Joint_Cosine'] = dot / (norm_s * norm_t + 1e-9)
    
    features = df_merged[['Frame_s', 'Joint_Name', 'Visibility_s', 'Joint_Error', 'Joint_Cosine', 'Label_s']].copy()
    features.rename(columns={'Frame_s': 'Frame', 'Visibility_s': 'Visibility', 'Label_s': 'Label'}, inplace=True)
    return features

**3. NẠP TOÀN BỘ 7 VIDEO DỮ LIỆU**

In [227]:
dance_pairs = [
   {"id": 1, "student": "student_dance_1.csv", "teacher": "teacher_dance_1.csv"},
    {"id": 2, "student": "student_dance_2.csv", "teacher": "teacher_dance_2.csv"},
    {"id": 3, "student": "student_dance_3.csv", "teacher": "teacher_dance_3.csv"},
    {"id": 4, "student": "student_dance_4.csv", "teacher": "teacher_dance_4.csv"},
    {"id": 5, "student": "student_dance_5.csv", "teacher": "teacher_dance_5.csv"},
  #  {"id": 6, "student": "student_dance_6.csv", "teacher": "teacher_dance_6.csv"}, 
  #  {"id": 7, "student": "student_dance_7.csv", "teacher": "teacher_dance_7.csv"},
   {"id": 8, "student": "student_dance_8.csv", "teacher": "teacher_dance_8.csv"},
  #  {"id": 9, "student": "student_dance_9.csv", "teacher": "teacher_dance_9.csv"},
    {"id": 10, "student": "student_dance_10.csv", "teacher": "teacher_dance_10.csv"}
]

all_extracted_features = []

for pair in dance_pairs:
    print(f"Đang xử lý Video_ID {pair['id']}...")
    try:
        df_s = pd.read_csv(pair["student"])
        df_t = pd.read_csv(pair["teacher"])
        
        df_s_clean = preprocess_data(df_s)
        df_t_clean = preprocess_data(df_t)
        
        features = align_and_extract_features(df_s_clean, df_t_clean)
        features['Video_ID'] = pair['id']
        all_extracted_features.append(features)
    except Exception as e:
        print(f"  -> Bỏ qua Video_ID {pair['id']}: Lỗi {e}")

if len(all_extracted_features) > 0:
    final_master_dataset = pd.concat(all_extracted_features, ignore_index=True)
    print("\nHoàn tất trích xuất đặc trưng!")

Đang xử lý Video_ID 1...
Đang xử lý Video_ID 2...
Đang xử lý Video_ID 3...
Đang xử lý Video_ID 4...
Đang xử lý Video_ID 5...
Đang xử lý Video_ID 8...
Đang xử lý Video_ID 10...

Hoàn tất trích xuất đặc trưng!


**4. HUẤN LUYỆN (SMOTEENN + TUNING + THRESHOLD) VÀ ĐÁNH GIÁ**

In [228]:
# One-Hot Encoding cho Joint_Name
df_encoded = pd.get_dummies(final_master_dataset, columns=['Joint_Name'])

TARGET_TEST_VIDEO_ID = [10,1,2,4]  # Chọn các video để test

train_data = df_encoded[~df_encoded['Video_ID'].isin(TARGET_TEST_VIDEO_ID)]
test_data = df_encoded[df_encoded['Video_ID'].isin(TARGET_TEST_VIDEO_ID)]

joint_cols = [col for col in df_encoded.columns if col.startswith('Joint_Name_')]
features_list = ['Joint_Error', 'Joint_Cosine'] + joint_cols

X_train = train_data[features_list]
y_train = train_data['Label']
X_test = test_data[features_list]
y_test = test_data['Label']

# [ĐÃ SỬA 1]: Áp dụng SMOTEENN để nhân bản dữ liệu lỗi VÀ dọn dẹp điểm nhiễu giáp ranh
print(f"Đang chạy SMOTEENN để làm sạch ranh giới dữ liệu Train...")
smote_enn = SMOTEENN(random_state=42)
X_train_res, y_train_res = smote_enn.fit_resample(X_train, y_train)
print(f"Số lượng điểm khớp sau xử lý: {len(X_train_res)}")

# [ĐÃ SỬA 2]: Huấn luyện RF với min_samples_leaf=15, BỎ class_weight='balanced'
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    min_samples_leaf=15, # Chống học vẹt
    random_state=42
)
rf_model.fit(X_train_res, y_train_res)

# [ĐÃ SỬA 3]: Ép Ngưỡng Khắt Khe (Strict Threshold)
# Thay vì dùng rf_model.predict(), ta lấy xác suất rơi vào Class 0
proba_class_0 = rf_model.predict_proba(X_test)[:, 0]

# Ép máy tính: Chỉ khi chắc chắn >= 80% là sai, mới được phạt (Giảm False Positive)
strict_threshold = 0.5
joint_predictions = np.where(proba_class_0 >= strict_threshold, 0, 1)

print(f"\n=== KẾT QUẢ ĐÁNH GIÁ CẤP ĐỘ KHỚP (TEST ID={TARGET_TEST_VIDEO_ID}) ===")
print(classification_report(y_test, joint_predictions))

# Tính toán F-PASS cấp độ Khung hình
test_data_eval = test_data.copy()
test_data_eval['Predicted_Label'] = joint_predictions

frame_summary = test_data_eval.groupby('Frame').agg(
    Total_Joints=('Predicted_Label', 'count'),
    Correct_Joints_Pred=('Predicted_Label', 'sum')
)

# Giữ điều kiện Đạt của Frame ở mức >= 10 khớp
frame_summary['Frame_Pass_Pred'] = frame_summary['Correct_Joints_Pred'] >= 9

F_pass = frame_summary['Frame_Pass_Pred'].sum()
F_total = len(frame_summary)
F_ratio = (F_pass / F_total) * 100 if F_total > 0 else 0

print(f"\n=== TÍNH TOÁN F-PASS (FRAME-LEVEL) ===")
print(f"- Tổng số Frame thực tế của video (F_total) : {F_total}")
print(f"- Số lượng Frame đạt chuẩn (>= 10 khớp) (F_pass) : {F_pass}")
print(f"- Tỷ lệ đúng động tác (F_pass / F_total)        : {F_ratio:.2f}%")

# Hiển thị Top Đặc trưng
importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Đặc trưng': features_list, 
    'Mức độ đóng góp (%)': importances * 100
})
print("\n=== TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT ===")
display(feature_imp_df.sort_values(by='Mức độ đóng góp (%)', ascending=False).head(10))

Đang chạy SMOTEENN để làm sạch ranh giới dữ liệu Train...
Số lượng điểm khớp sau xử lý: 11745

=== KẾT QUẢ ĐÁNH GIÁ CẤP ĐỘ KHỚP (TEST ID=[10, 1, 2, 4]) ===
              precision    recall  f1-score   support

           0       0.57      0.61      0.59     16711
           1       0.61      0.57      0.59     17752

    accuracy                           0.59     34463
   macro avg       0.59      0.59      0.59     34463
weighted avg       0.59      0.59      0.59     34463


=== TÍNH TOÁN F-PASS (FRAME-LEVEL) ===
- Tổng số Frame thực tế của video (F_total) : 1041
- Số lượng Frame đạt chuẩn (>= 10 khớp) (F_pass) : 672
- Tỷ lệ đúng động tác (F_pass / F_total)        : 64.55%

=== TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT ===


,Đặc trưng,Mức độ đóng góp (%)
0,Joint_Error,41.459832
1,Joint_Cosine,35.796660
7,Joint_Name_left_wrist,2.253591
6,Joint_Name_left_shoulder,2.202584
2,Joint_Name_left_ankle,2.186058
14,Joint_Name_right_wrist,2.136543
5,Joint_Name_left_knee,1.937750
12,Joint_Name_right_knee,1.923786
3,Joint_Name_left_elbow,1.874432
8,Joint_Name_nose,1.785657
